# Imports

In [1]:
import os
import json

from groq import Groq
from dotenv import load_dotenv
from pathlib import Path

# Params

In [2]:
PROJECT_ROOT = Path.cwd().parent

OUTPUT_DIR = PROJECT_ROOT / "data" / "raw"

USER_INPUT_PATH = (
    OUTPUT_DIR
    / "user_input.json"
)

In [3]:
MODEL = "llama-3.1-8b-instant"

In [4]:
with USER_INPUT_PATH.open(
    encoding="utf-8",
) as file:
    user_input = json.load(file)

USER_TOPIC = user_input["research_topic"]

print(USER_TOPIC)

i want to investigate about covid-19 vaccine validation


# Prompt and Response

In [5]:
prompt = f"""
You are an expert AI researcher.

Your task is to generate an optimized search query for the arXiv API.

The query must follow the official arXiv query syntax.

Rules:

- Use only English keywords.
- Use the field "all:" for every search term or phrase.
- Boolean operators (AND, OR) must always be OUTSIDE quotation marks.
- Every quoted phrase must have its own "all:" prefix.
- Group OR conditions using parentheses.
- Generate a query that maximizes relevant papers while avoiding unrelated ones.
- Keep the query concise.

Examples:

User topic:
"Multi-agent AI for data analytics"

Correct output:
(all:"multi-agent" OR all:"agentic AI") AND all:"data analytics"

User topic:
"Gaussian distributions in probability theory"

Correct output:
(all:"gaussian distribution" OR all:"gaussian function") AND all:"probability theory"

User topic:
"LLM agents for software engineering"

Correct output:
(all:"LLM" OR all:"large language model") AND all:"software engineering"

Before generating the final query:

1. Identify the main concepts.
2. Generate useful synonyms.
3. Build the arXiv query.

Do NOT show your reasoning.
Return ONLY the final query.

User topic:

{USER_TOPIC}
"""

In [6]:
load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

In [7]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ]
)

In [8]:
answer = response.choices[0].message.content

print(answer)

(all:"Covid-19" OR all:"SARS-CoV-2") AND (all:"vaccine validation" OR all:"vaccine efficacy")


# Save processed Query

In [9]:
config = {
    "topic": USER_TOPIC.strip(),
    "query": answer,
}

with open(
    OUTPUT_DIR / "arxiv_query.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        config,
        file,
        indent=4,
        ensure_ascii=False,
    )